In [1]:
"""
SCA-U2net regression pipeline (model + training).

Reported performance: 5-fold GroupKFold CV RMSE (TVT) = 9.5521.

Architecture adapted from Wang et al. (2024) SCA-U2net (semantic segmentation)
to sequence-to-sequence regression. Multi-scale nested U-shape with spatial/channel
attention + deep supervision via 7-output masked RMSE.

Input:  [B, in_ch, L]  (in_ch=56, L=window size)
Output: [B, L] fuse prediction + list of 6 side outputs [B, L]
"""

import os
os.environ["PYTHONHASHSEED"] = "42"

import math
import hashlib
import warnings
import time
import gc
import multiprocessing
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import joblib
from numba import njit
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# =============================================================================
# Configuration
# =============================================================================

class CFG:
    dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    artifacts_path = Path("/kaggle/input/datasets/bioconvolutionyt/sca-u2net-artifacts")
    n_splits = 5
    cv = GroupKFold(n_splits=5)
    metric = root_mean_squared_error

SEED = 8
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False

IS_KAGGLE = Path("/kaggle/working").exists()
NCPU = min(8, multiprocessing.cpu_count())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Physical pipeline scales
PF_SCALES = (3.0, 5.0, 8.0, 12.0)

# Beam configs (7 beams)
BEAMS = [
    (10, 20.0, 144.0, 2, "cons"),
    (10,  8.0,  64.0, 2, "loose"),
    ( 8, 35.0, 220.0, 1, "vcons"),
    (10, 14.0,  90.0, 5, "sm5"),
    (20,  4.0,  36.0, 3, "vloose"),
    (12, 12.0, 100.0, 3, "mid"),
    (15, 25.0, 180.0, 2, "stiff"),
]

# PF params
PF_N = 600; ANCC_N = 600
PF_GR_SIG_MIN = 10.; PF_GR_SIG_MAX = 60.; PF_GR_SIG_DEF = 30.
PF_RESAMP = 0.5
C4B_PF_MOM = 0.993; C4B_PF_VN = 0.005; C4B_PF_PN = 0.01
ANCC_ALPHA = 0.998; ANCC_RN = 0.002; ANCC_PN = 0.005
ANCC_IR = 0.01; ANCC_IS = 0.3; ANCC_RP = 0.1; ANCC_RR = 0.001
PF_INIT_V_STD = 0.02; PF_ROUGH_P = 0.2; PF_ROUGH_V = 0.003
PF_GR_WIN = 5; PF_GR_WT = 0.3

PF_ENS_SEEDS = 16
PF_ENS_PARTICLES = 150
PF_ENS_SCALE = 5.0

# DTW params
DTW_RADII = (20, 50, 100, 200)
DTW_STOCH_K = 12
DTW_STOCH_TEMP = 3.0

# Spatial / Formation
FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10; DENSE_SPW = 60; DENSE_K = 20; N_SPLITS = 5

# lik-PF parameters (128-seed likelihood-weighted particle filter)
LIKPF_SEEDS = 128
LIKPF_PARTICLES = 500
LIKPF_SCALES = (3.0, 5.0, 8.0, 12.0)
LIKPF_INIT_SPREAD = 4.5
LIKPF_MOM = 0.998
LIKPF_VN = 0.002
LIKPF_PN = 0.005
LIKPF_RP = 0.1
LIKPF_RR = 0.001
LIKPF_RESAMP = 0.5

# Artifact paths
FEAT_CACHE = str(CFG.artifacts_path / "sca_u2net_features_cache.pkl")
MODEL_DIR = str(CFG.artifacts_path / "models_sca_u2net")
NORM_FILE = str(CFG.artifacts_path / "sca_u2net_norm_stats.pkl")

# =============================================================================
# SCA-U2Net Model Architecture
# =============================================================================


@dataclass
class UNetRegConfig:
    in_ch: int = 56
    mid_ch: int = 16
    out_ch: int = 64
    sam_kernel: int = 7
    cam_reduction: int = 16

# =============================================================================
# Building Blocks
# =============================================================================

class REBNCONV(nn.Module):
    """Conv1d + BatchNorm + ReLU (kernel=3, same padding via dilation)."""

    def __init__(self, in_ch: int, out_ch: int, dilation: int = 1):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(self.bn(self.conv(x)))


def _upsample_like(x: torch.Tensor, target_len: int) -> torch.Tensor:
    """Interpolate 1D feature [B, C, L'] to target_len."""
    return F.interpolate(x, size=target_len, mode="linear", align_corners=False)


class RSU(nn.Module):
    """Residual U-block with pooling (heights 7/6/5/4 for E1-E4)."""

    def __init__(self, height: int, in_ch: int, mid_ch: int, out_ch: int):
        super().__init__()
        assert height >= 2
        self.height = height
        self.rebnconvin = REBNCONV(in_ch, out_ch)

        self.enc = nn.ModuleList()
        self.enc.append(REBNCONV(out_ch, mid_ch))
        for _ in range(1, height - 1):
            self.enc.append(REBNCONV(mid_ch, mid_ch))

        self.bottleneck = REBNCONV(mid_ch, mid_ch, dilation=2)

        self.dec = nn.ModuleList()
        for _ in range(height - 2):
            self.dec.append(REBNCONV(mid_ch * 2, mid_ch))
        self.dec_last = REBNCONV(mid_ch * 2, out_ch)

        self.pool = nn.MaxPool1d(2, stride=2, ceil_mode=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hxin = self.rebnconvin(x)

        enc_feats = [self.enc[0](hxin)]
        for i in range(1, self.height - 1):
            down = self.pool(enc_feats[-1])
            enc_feats.append(self.enc[i](down))

        hx = self.bottleneck(enc_feats[-1])

        hx = self.dec[0](torch.cat([hx, enc_feats[-1]], dim=1))
        for i in range(1, self.height - 1):
            skip = enc_feats[self.height - 2 - i]
            hx = _upsample_like(hx, skip.shape[-1])
            if i < self.height - 2:
                hx = self.dec[i](torch.cat([hx, skip], dim=1))
            else:
                hx = self.dec_last(torch.cat([hx, skip], dim=1))

        return hx + hxin


class RSU4F(nn.Module):
    """Residual U-block with dilated convolutions (no pooling, for E5/E6/D5)."""

    def __init__(self, in_ch: int, mid_ch: int, out_ch: int):
        super().__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch)
        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dilation=1)
        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dilation=2)
        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dilation=4)
        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dilation=8)
        self.rebnconv3d = REBNCONV(mid_ch * 2, mid_ch, dilation=4)
        self.rebnconv2d = REBNCONV(mid_ch * 2, mid_ch, dilation=2)
        self.rebnconv1d = REBNCONV(mid_ch * 2, out_ch, dilation=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hxin = self.rebnconvin(x)
        hx1 = self.rebnconv1(hxin)
        hx2 = self.rebnconv2(hx1)
        hx3 = self.rebnconv3(hx2)
        hx4 = self.rebnconv4(hx3)
        hx3d = self.rebnconv3d(torch.cat([hx4, hx3], dim=1))
        hx2d = self.rebnconv2d(torch.cat([hx3d, hx2], dim=1))
        hx1d = self.rebnconv1d(torch.cat([hx2d, hx1], dim=1))
        return hx1d + hxin


# =============================================================================
# Attention Modules
# =============================================================================

class SAM(nn.Module):
    """Spatial Attention Module (for shallow encoders E1-E4)."""

    def __init__(self, channels: int, kernel_size: int = 7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv1d(2, channels, kernel_size=kernel_size, padding=padding)
        self.sigmoid = nn.Sigmoid()

    def forward(self, y1: torch.Tensor) -> torch.Tensor:
        max_c = torch.max(y1, dim=1, keepdim=True)[0]
        avg_c = torch.mean(y1, dim=1, keepdim=True)
        attn = self.conv(torch.cat([max_c, avg_c], dim=1))
        return self.sigmoid(attn) * y1


class CAM(nn.Module):
    """Channel Attention Module (for deep encoders E5/E6)."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, y1: torch.Tensor) -> torch.Tensor:
        max_s = torch.max(y1, dim=2)[0]
        avg_s = torch.mean(y1, dim=2)
        attn = self.mlp(max_s) + self.mlp(avg_s)
        return self.sigmoid(attn).unsqueeze(-1) * y1


# =============================================================================
# Attention-Enhanced Encoder Modules
# =============================================================================

class USAE(nn.Module):
    """U-shaped Spatial Attention Encoding (E1-E4): RSU + SAM, dual output."""

    def __init__(self, height: int, in_ch: int, mid_ch: int, out_ch: int, sam_kernel: int = 7):
        super().__init__()
        self.rsu = RSU(height, in_ch, mid_ch, out_ch)
        self.sam = SAM(out_ch, kernel_size=sam_kernel)

    def forward(self, x: torch.Tensor):
        y1 = self.rsu(x)
        y2 = self.sam(y1)
        return y1, y2


class UCAE(nn.Module):
    """U-shaped Channel Attention Encoding (E5/E6): RSU4F + CAM."""

    def __init__(self, in_ch: int, mid_ch: int, out_ch: int, reduction: int = 16, bottom: bool = False):
        super().__init__()
        self.rsu = RSU4F(in_ch, mid_ch, out_ch)
        self.cam = CAM(out_ch, reduction=reduction)
        self.bottom = bottom

    def forward(self, x: torch.Tensor):
        y1 = self.rsu(x)
        y2 = self.cam(y1)
        if self.bottom:
            return y2
        return y1, y2


# =============================================================================
# SCA-U2Net Regression Model
# =============================================================================

class SCA_U2Net_Reg(nn.Module):
    """SCA-U2Net adapted for per-point regression (delta prediction).

    Input:  x [B, in_ch, L], mask [B, L] (optional, for loss only)
    Output: (fuse_out [B, L], side_outputs list of 6x [B, L])

    Architecture:
      E1-E4: USAE (RSU + SAM) with progressive downsampling
      E5-E6: UCAE (RSU4F + CAM) with dilated convolutions
      D5-D1: RSU/RSU4F decoders with skip connections from C_out2
      6 side outputs + 1 fused output -> per-point regression
    """

    def __init__(self, cfg: UNetRegConfig = None):
        super().__init__()
        cfg = cfg or UNetRegConfig()
        c, m, o = cfg.in_ch, cfg.mid_ch, cfg.out_ch
        sk, r = cfg.sam_kernel, cfg.cam_reduction

        self.e1 = USAE(7, c, m, o, sam_kernel=sk)
        self.e2 = USAE(6, o, m, o, sam_kernel=sk)
        self.e3 = USAE(5, o, m, o, sam_kernel=sk)
        self.e4 = USAE(4, o, m, o, sam_kernel=sk)
        self.e5 = UCAE(o, m, o, reduction=r, bottom=False)
        self.e6 = UCAE(o, m, o, reduction=r, bottom=True)

        self.d5 = RSU4F(o * 2, m, o)
        self.d4 = RSU(4, o * 2, m, o)
        self.d3 = RSU(5, o * 2, m, o)
        self.d2 = RSU(6, o * 2, m, o)
        self.d1 = RSU(7, o * 2, m, o)

        self.pool = nn.MaxPool1d(2, stride=2, ceil_mode=True)

        self.side1 = nn.Conv1d(o, 1, kernel_size=3, padding=1)
        self.side2 = nn.Conv1d(o, 1, kernel_size=3, padding=1)
        self.side3 = nn.Conv1d(o, 1, kernel_size=3, padding=1)
        self.side4 = nn.Conv1d(o, 1, kernel_size=3, padding=1)
        self.side5 = nn.Conv1d(o, 1, kernel_size=3, padding=1)
        self.side6 = nn.Conv1d(o, 1, kernel_size=3, padding=1)

        self.outconv = nn.Conv1d(6, 1, kernel_size=1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, list[torch.Tensor]]:
        """
        Args:
            x: [B, in_ch, L] input features (channel-first)
        Returns:
            fuse_out: [B, L] fused regression output
            sides: list of 6 tensors, each [B, L]
        """
        e1c1, e1c2 = self.e1(x)
        e2c1, e2c2 = self.e2(self.pool(e1c1))
        e3c1, e3c2 = self.e3(self.pool(e2c1))
        e4c1, e4c2 = self.e4(self.pool(e3c1))
        e5c1, e5c2 = self.e5(self.pool(e4c1))
        e6c = self.e6(self.pool(e5c1))

        hx6up = _upsample_like(e6c, e5c2.shape[-1])
        hx5d = self.d5(torch.cat([hx6up, e5c2], dim=1))
        hx5dup = _upsample_like(hx5d, e4c2.shape[-1])
        hx4d = self.d4(torch.cat([hx5dup, e4c2], dim=1))
        hx4dup = _upsample_like(hx4d, e3c2.shape[-1])
        hx3d = self.d3(torch.cat([hx4dup, e3c2], dim=1))
        hx3dup = _upsample_like(hx3d, e2c2.shape[-1])
        hx2d = self.d2(torch.cat([hx3dup, e2c2], dim=1))
        hx2dup = _upsample_like(hx2d, e1c2.shape[-1])
        hx1d = self.d1(torch.cat([hx2dup, e1c2], dim=1))

        L = x.shape[-1]
        d1 = self.side1(hx1d)
        d2 = _upsample_like(self.side2(hx2d), L)
        d3 = _upsample_like(self.side3(hx3d), L)
        d4 = _upsample_like(self.side4(hx4d), L)
        d5 = _upsample_like(self.side5(hx5d), L)
        d6 = _upsample_like(self.side6(e6c), L)

        fuse = self.outconv(torch.cat([d1, d2, d3, d4, d5, d6], dim=1))

        fuse_out = fuse.squeeze(1)
        sides = [d1.squeeze(1), d2.squeeze(1), d3.squeeze(1),
                 d4.squeeze(1), d5.squeeze(1), d6.squeeze(1)]

        return fuse_out, sides


# =============================================================================
# Deep Supervision Loss (Masked RMSE)
# =============================================================================

def deep_supervision_rmse(
    fuse: torch.Tensor,
    sides: list[torch.Tensor],
    target: torch.Tensor,
    mask: torch.Tensor,
    eps: float = 1e-8,
) -> torch.Tensor:
    """Deep supervision loss: sum of masked RMSE over fuse + 6 side outputs.

    Args:
        fuse: [B, L] fused prediction
        sides: list of 6 tensors [B, L]
        target: [B, L] ground truth delta
        mask: [B, L] binary mask (1=valid, 0=padding)
    """
    def _masked_rmse(pred, tgt, m):
        valid = m.float()
        n_valid = valid.sum()
        if n_valid < 1:
            return torch.tensor(0.0, device=pred.device, requires_grad=True)
        diff2 = ((pred - tgt) ** 2) * valid
        return torch.sqrt(diff2.sum() / n_valid + eps)

    total = _masked_rmse(fuse, target, mask)
    for s in sides:
        total = total + _masked_rmse(s, target, mask)
    return total




@dataclass
class TrainConfig:
    window_size: int = 512
    stride: int = 256
    batch_size: int = 32
    epochs: int = 50
    lr: float = 1e-3
    weight_decay: float = 1e-4
    patience: int = 7
    n_splits: int = 5
    seed: int = 49

UCFG = UNetRegConfig()
TRCFG = TrainConfig()

# =============================================================================
# SECTION 2: Numba JIT Functions (physical priors)
# =============================================================================

@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t

@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1]=cum[j]+w[j]
    u0=np.random.uniform(0.,1./N)
    np2=np.empty(N); na=np.empty(N); ci=0
    for j in range(N):
        u=u0+j/N
        while ci<N-1 and cum[ci+1]<u: ci+=1
        np2[j]=pos[ci]+rp*np.random.randn()
        na[j] =aux[ci]+rv*np.random.randn()
    return np2,na

@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]
                cC[i],cC[mi]=cC[mi],cC[i]
                cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

@njit(cache=True)
def _beam_jit_weighted(sgr, tw_gr, si, BS, mc, es, alpha):
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        w_step=alpha+(1.0-alpha)*(step/max(n-1,1))
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2*w_step/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]
                cC[i],cC[mi]=cC[mi],cC[i]
                cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path

@njit(cache=True)
def _dtw_sakoe_chiba(query, ref, radius):
    N = len(query); M = len(ref)
    INF = 1e18
    D = np.full((N, M), INF)
    slope = (M - 1.0) / max(N - 1.0, 1.0)
    for i in range(N):
        j_center = int(round(i * slope))
        j_lo = max(0, j_center - radius)
        j_hi = min(M - 1, j_center + radius)
        for j in range(j_lo, j_hi + 1):
            cost = (query[i] - ref[j]) ** 2
            if i == 0 and j == 0:
                D[i, j] = cost
            elif i == 0:
                prev = D[i, j - 1]
                D[i, j] = cost + (prev if prev < INF else INF)
            elif j == 0:
                prev = D[i - 1, j]
                D[i, j] = cost + (prev if prev < INF else INF)
            else:
                a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
                mn = a if a < b else b
                mn = mn if mn < c else c
                D[i, j] = cost + (mn if mn < INF else INF)
    i = N - 1; j = M - 1
    pi = np.zeros(N + M, np.int64)
    pj = np.zeros(N + M, np.int64)
    k = 0
    while i > 0 or j > 0:
        pi[k] = i; pj[k] = j; k += 1
        if i == 0: j -= 1
        elif j == 0: i -= 1
        else:
            a = D[i-1,j-1]; b = D[i-1,j]; c = D[i,j-1]
            if a <= b and a <= c: i -= 1; j -= 1
            elif b <= c: i -= 1
            else: j -= 1
    pi[k] = 0; pj[k] = 0; k += 1
    return D, pi[:k], pj[:k]

@njit(cache=True)
def _dtw_path_to_tvt(pi, pj, tw_tvt, N):
    j_for_i = np.zeros(N, np.int64)
    for k in range(len(pi)):
        j_for_i[pi[k]] = pj[k]
    result = np.empty(N, np.float32)
    for i in range(N):
        result[i] = tw_tvt[j_for_i[i]]
    return result

@njit(cache=True)
def _dtw_path_slope(pi, pj, N, smooth_win=5):
    j_for_i = np.zeros(N, np.float64)
    for k in range(len(pi)):
        j_for_i[pi[k]] = float(pj[k])
    slope = np.zeros(N, np.float32)
    hw = smooth_win // 2
    for i in range(N):
        i0 = max(0, i - hw); i1 = min(N - 1, i + hw)
        if i1 > i0:
            slope[i] = float((j_for_i[i1] - j_for_i[i0]) / (i1 - i0))
        else:
            slope[i] = 1.0
    return slope

@njit(cache=True)
def _dtw_stochastic_realizations(query, ref, radius, K, temperature, jit_seed=0):
    np.random.seed(jit_seed)
    N = len(query); M = len(ref)
    INF = 1e18
    slope = (M - 1.0) / max(N - 1.0, 1.0)
    D_base = np.full((N, M), INF)
    for i in range(N):
        j_center = int(round(i * slope))
        j_lo = max(0, j_center - radius)
        j_hi = min(M - 1, j_center + radius)
        for j in range(j_lo, j_hi + 1):
            D_base[i, j] = (query[i] - ref[j]) ** 2
    paths = np.zeros((K, N), np.int64)
    for k in range(K):
        D = np.full((N, M), INF)
        for i in range(N):
            j_center = int(round(i * slope))
            j_lo = max(0, j_center - radius)
            j_hi = min(M - 1, j_center + radius)
            for j in range(j_lo, j_hi + 1):
                noise = -temperature * np.log(-np.log(np.random.uniform(1e-10, 1.0)))
                cost = D_base[i, j] + noise
                if i == 0 and j == 0: D[i, j] = cost
                elif i == 0:
                    prev = D[i, j-1]
                    D[i, j] = cost + (prev if prev < INF else INF)
                elif j == 0:
                    prev = D[i-1, j]
                    D[i, j] = cost + (prev if prev < INF else INF)
                else:
                    a = D[i-1,j-1]; b = D[i-1,j]; c = D[i,j-1]
                    mn = a if a < b else b
                    mn = mn if mn < c else c
                    D[i, j] = cost + (mn if mn < INF else INF)
        i = N-1; j = M-1
        j_for_i = np.zeros(N, np.int64)
        while i > 0 or j > 0:
            j_for_i[i] = j
            if i == 0: j -= 1
            elif j == 0: i -= 1
            else:
                a = D[i-1,j-1]; b = D[i-1,j]; c = D[i,j-1]
                if a <= b and a <= c: i -= 1; j -= 1
                elif b <= c: i -= 1
                else: j -= 1
        j_for_i[0] = j
        paths[k] = j_for_i
    return paths

@njit(cache=True)
def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
              ALPHA,RN,PN,IS,RP,RR,RESAMP,jit_seed=0):
    np.random.seed(jit_seed)
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+0.01*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=(gr_v[i]-eg)/gs
                lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_

@njit(cache=True)
def _pf_ancc_loglik(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
                    ALPHA,RN,PN,IS,RP,RR,RESAMP,jit_seed=0):
    np.random.seed(jit_seed)
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+0.01*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    log_lik=0.0
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            avg_lk=0.0; ws=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=(gr_v[i]-eg)/gs
                lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                avg_lk+=w[j]*lk
                w[j]*=lk; ws+=w[j]
            log_lik+=np.log(max(avg_lk,1e-300))
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_,log_lik

@njit(cache=True)
def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
          gs,ip,iv,beta,icpt,zsig,N,
          MOM,VN,PN,GR_WT,RP,RV,RESAMP,jit_seed=0):
    np.random.seed(jit_seed)
    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ip+0.5*np.random.randn()
        vel[j]=iv+0.02*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        for j in range(N):
            vel[j]=MOM*vel[j]+VN*np.random.randn()
            pos[j]+=vel[j]*dm+PN*np.random.randn()
            pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                ep=_interp1(gg_p,pos[j],vmin,step)
                dp=(gr_v[i]-ep)/gs
                lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es=_interp1(gg_s,pos[j],vmin,step)
                    ds=(gr_sm_v[i]-es)/(gs*1.5)
                    ls_v=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                    lk=(1.-GR_WT)*lp+GR_WT*ls_v
                else: lk=lp
                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ws2=0.
        for j in range(N):
            dv=(vel[j]-ve)/max(zsig*2.,0.005)
            lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
            w[j]*=lz; ws2+=w[j]
        if ws2>0.:
            for j in range(N): w[j]/=ws2
        else:
            for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,vel=_resamp(pos,vel,w,N,RP,RV)
            for j in range(N): w[j]=1./N
        wm=0.
        for j in range(N): wm+=w[j]*pos[j]
        pts[i]=wm; va=0.
        for j in range(N): va+=w[j]*(pos[j]-wm)**2
        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
    return pts,std_

@njit(cache=True, nogil=True)
def _pf_lik_allseeds(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N, n_seeds, seed_base,
                     MOM, VN, PN, RP, RR, RESAMP, init_spr):
    n = len(md_v)
    preds = np.empty((n_seeds, n))
    liks = np.empty(n_seeds)
    tmax = vmin + len(gg) * step
    for s in range(n_seeds):
        np.random.seed(seed_base + s)
        pos = np.empty(N); rate = np.empty(N); w = np.ones(N) / N
        for j in range(N):
            pos[j] = ls + init_spr * np.random.randn()
            rate[j] = ir + 0.01 * np.random.randn()
        log_lik = 0.0; prev_md = md_v[0] - 1.0
        for i in range(n):
            dm = md_v[i] - prev_md
            if dm < 1.0: dm = 1.0
            for j in range(N):
                rate[j] = MOM * rate[j] + VN * np.random.randn()
                pos[j] += rate[j] * dm + PN * np.random.randn()
                tvt_j = pos[j] - z_v[i]
                if tvt_j < vmin - 100.: tvt_j = vmin - 100.
                if tvt_j > tmax + 100.: tvt_j = tmax + 100.
                pos[j] = tvt_j + z_v[i]
            avg_lk = 0.0
            for j in range(N):
                eg = _interp1(gg, pos[j] - z_v[i], vmin, step)
                d = (gr_v[i] - eg) / gs
                dd = d * d
                if dd > 600.: dd = 600.
                lk = np.exp(-0.5 * dd)
                if lk < 1e-300: lk = 1e-300
                avg_lk += w[j] * lk
                w[j] = w[j] * lk
            if avg_lk < 1e-300: avg_lk = 1e-300
            log_lik += np.log(avg_lk)
            ws = 0.0
            for j in range(N): ws += w[j]
            if ws > 0.0:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
            neff = 0.0
            for j in range(N): neff += w[j] * w[j]
            neff = 1.0 / neff
            if neff < RESAMP * N:
                cum = np.empty(N); c = 0.0
                for j in range(N): c += w[j]; cum[j] = c
                u0 = np.random.uniform(0., 1. / N)
                newpos = np.empty(N); newrate = np.empty(N); ci = 0
                for j in range(N):
                    u = u0 + j / N
                    while ci < N - 1 and cum[ci] < u: ci += 1
                    newpos[j] = pos[ci] + RP * np.random.randn()
                    newrate[j] = rate[ci] + RR * np.random.randn()
                for j in range(N):
                    pos[j] = newpos[j]; rate[j] = newrate[j]; w[j] = 1. / N
            est = 0.0
            for j in range(N): est += w[j] * (pos[j] - z_v[i])
            preds[s, i] = est; prev_md = md_v[i]
        liks[s] = log_lik
    return preds, liks


# =============================================================================
# SECTION 3: Python Wrapper Functions for Physical Algorithms
# =============================================================================

def _grid(tw_tvt, tw_gr, step=0.2):
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
    tvt_g=np.arange(tmin,tmax+step,step)
    return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)

def _gr_sig(hw, tw_tvt, tw_gr):
    kn=hw[hw['TVT_input'].notna()&hw['GR'].notna()]
    if len(kn)<20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn['GR'].values-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)),
                          PF_GR_SIG_MIN,PF_GR_SIG_MAX))

def _nn(arr, v):
    i=int(np.searchsorted(arr,v,'left'))
    if i>=len(arr): return len(arr)-1
    if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
    return i

def _smooth(vals, fb, r):
    s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)

def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si=_nn(tw_tvt,start_tvt)
    sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
    path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
    return tw_tvt[path].astype(np.float32)

def beam_search_weighted(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r, alpha=0.3):
    si=_nn(tw_tvt,start_tvt)
    sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
    path=_beam_jit_weighted(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es),float(alpha))
    return tw_tvt[path].astype(np.float32)

def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N, jit_seed=0):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    pts,std=_pf_ancc(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                      ev['GR'].values.astype(np.float64),gg,gmin,gst,
                      gs,ls,ir,N,ANCC_ALPHA,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP,
                      jit_seed)
    return pts.astype(np.float32),std.astype(np.float32)

def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N, jit_seed=0):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values.astype(np.float32)
    kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
    dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
    if m2.sum()>=10:
        vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
        A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
    else: beta,icpt,zsig=-1.,0.,0.1
    t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
    iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    gs2,_,_=_grid(tw_tvt,tw_s)
    gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
    pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                   ev['GR'].values.astype(np.float64),
                   gr_sm.loc[ev.index].values.astype(np.float64),
                   gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                   beta,icpt,zsig,N,
                   C4B_PF_MOM,C4B_PF_VN,C4B_PF_PN,PF_GR_WT,PF_ROUGH_P,PF_ROUGH_V,PF_RESAMP,
                   jit_seed)
    return pts.astype(np.float32),std.astype(np.float32)

def run_pf_ancc_loglik(hw, tw_tvt, tw_gr, N=PF_ENS_PARTICLES, jit_seed=0):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([]),0.0
    ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    pts,std,ll=_pf_ancc_loglik(
        ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
        ev['GR'].values.astype(np.float64),gg,gmin,gst,
        gs,ls,ir,N,ANCC_ALPHA,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP,
        jit_seed)
    return pts.astype(np.float32),std.astype(np.float32),float(ll)

def run_pf_lik_ensemble(hw, tw_tvt, tw_gr, n_seeds=PF_ENS_SEEDS,
                        n_particles=PF_ENS_PARTICLES, scale=PF_ENS_SCALE, base_seed=0):
    preds=[]; logliks=[]
    for s in range(n_seeds):
        seed_s=(base_seed+s*7919)%(2**31)
        pts,_,ll=run_pf_ancc_loglik(hw,tw_tvt,tw_gr,N=n_particles,jit_seed=seed_s)
        if len(pts)==0: return np.array([]),np.array([]),0.0
        preds.append(pts); logliks.append(ll)
    logliks=np.array(logliks,dtype=np.float64)
    weights=np.exp((logliks-logliks.max())/scale); weights/=weights.sum()
    pred_stack=np.stack(preds,0)
    ensemble=(weights[:,None]*pred_stack).sum(0).astype(np.float32)
    ens_std=np.sqrt((weights[:,None]*(pred_stack-ensemble[None,:])**2).sum(0)).astype(np.float32)
    return ensemble,ens_std,float(logliks.max())

def run_dtw_multiscale(query_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII):
    N = len(query_gr)
    qn = (query_gr - query_gr.mean()) / (query_gr.std() + 1e-6)
    rn = (tw_gr - tw_gr.mean()) / (tw_gr.std() + 1e-6)
    qn_f = qn.astype(np.float64); rn_f = rn.astype(np.float64)
    dtw_tvts = {}; dtw_slopes = {}; dtw_costs = {}
    inv_cost_sum = 0.0; tvt_stack = []
    for r in radii:
        D, pi, pj = _dtw_sakoe_chiba(qn_f, rn_f, r)
        cost = float(D[len(qn_f)-1, len(rn_f)-1]) / max(len(qn_f)+len(rn_f), 1)
        tvt_pred = _dtw_path_to_tvt(pi[::-1], pj[::-1], tw_tvt.astype(np.float32), N)
        slope = _dtw_path_slope(pi[::-1], pj[::-1], N)
        dtw_tvts[r] = tvt_pred; dtw_slopes[r] = slope; dtw_costs[r] = cost
        ic = 1.0 / (cost + 1e-6); inv_cost_sum += ic
        tvt_stack.append((tvt_pred, ic))
    weights = np.array([ic / inv_cost_sum for _, ic in tvt_stack], dtype=np.float32)
    tvts_mat = np.stack([t for t, _ in tvt_stack], axis=1)
    dtw_ens = (tvts_mat * weights[None, :]).sum(axis=1).astype(np.float32)
    return dtw_tvts, dtw_slopes, dtw_costs, dtw_ens

def run_dtw_stochastic(query_gr, tw_tvt, tw_gr, last_tvt,
                       radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP, jit_seed=0):
    N = len(query_gr)
    qn = ((query_gr - query_gr.mean()) / (query_gr.std() + 1e-6)).astype(np.float64)
    rn = ((tw_gr - tw_gr.mean()) / (tw_gr.std() + 1e-6)).astype(np.float64)
    paths = _dtw_stochastic_realizations(qn, rn, radius, K, temperature, jit_seed)
    tvt_realiz = np.empty((K, N), dtype=np.float32)
    for k in range(K):
        for i in range(N):
            tvt_realiz[k, i] = tw_tvt[paths[k, i]]
    mean_tvt = tvt_realiz.mean(axis=0).astype(np.float32)
    std_tvt = tvt_realiz.std(axis=0).astype(np.float32)
    cv_tvt = (std_tvt / (np.abs(mean_tvt) + 1e-6)).astype(np.float32)
    return mean_tvt, std_tvt, cv_tvt

def robust_slope(x, y, w=None):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<2 or np.std(x[m])<1e-6: return 0.
    return float(np.polyfit(x[m],y[m],1)[0])

def seg_b_well(ktvt, kz, form_col):
    bv=ktvt+kz-form_col; n=len(bv)
    b_full=float(np.median(bv))
    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
    t1,t2=n//3, 2*n//3
    b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
    b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
    w=np.exp(0.02*np.arange(n)); w/=w.sum()
    b_wls=float(np.dot(w,bv))
    return b_full,b_early,b_mid,b_late,b_wls

def multi_scale_ncc(kgr, ktvt, hgr, hws=(8,15,25), stride=3):
    out=[]
    for hw in hws:
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0:
            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
    tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
    sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
    sc_ens=(tvts*sw).sum(1).astype(np.float32)
    return out, sc_ens

def lik_pf(hw, tw, n_particles=LIKPF_PARTICLES, n_seeds=LIKPF_SEEDS,
           scales=LIKPF_SCALES, init_spr=LIKPF_INIT_SPREAD, seed_base=0):
    """Likelihood-weighted PF ensemble (128-seed). Returns ({scale: pred}, ev_index)."""
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return {}, np.array([], dtype=int)
    last = kn.iloc[-1]
    ls = float(last['TVT_input']) + float(last['Z'])
    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))
    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values); dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0
    tmin = float(tw_tvt.min()); tmax_tw = float(tw_tvt.max())
    gstep = 0.2
    tvt_g = np.arange(tmin, tmax_tw + gstep, gstep)
    gg = np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64)
    gr_v = hw['GR'].interpolate(limit_direction="both").fillna(tw_gr.mean()).values.astype(float)[ev.index]
    preds, liks = _pf_lik_allseeds(
        ev['MD'].values.astype(float), ev['Z'].values.astype(float), gr_v,
        gg, tmin, gstep, gs, ls, ir, n_particles, n_seeds, seed_base,
        LIKPF_MOM, LIKPF_VN, LIKPF_PN, LIKPF_RP, LIKPF_RR, LIKPF_RESAMP, init_spr)
    ln = liks - liks.max()
    out = {}
    for sc in scales:
        wts = np.exp(ln / float(sc)); wts /= wts.sum()
        out[f"pf_scale_{sc:g}"] = (wts[:, None] * preds).sum(0).astype(np.float32)
    out["pf_mean"] = preds.mean(0).astype(np.float32)
    return out, ev.index.values


# =============================================================================
# SECTION 4: Spatial / Formation Imputers
# =============================================================================

class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows=[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
            except: continue
            if len(df)==0: continue
            row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
            rows.append(row)
        self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
        xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
        self.tree=cKDTree(xy/self.scale)
        self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
        self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q=xy_q/self.scale; nf=min(k+5,len(self.df))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
        ord_=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord_,1); ik=np.take_along_axis(idx,ord_,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
        xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
        A=np.zeros((len(q),3,3))
        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
        rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
        try: coef=np.linalg.solve(A,rhs)
        except:
            coef=np.zeros((len(q),3,6))
            for r in range(len(q)):
                try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                except: pass
        Xq=xy_q[:,0]; Yq=xy_q[:,1]
        pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
        pred[~vk.any(1)]=self.fa.mean(0)
        return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)


class DenseANCCImputer:
    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs,ys,anccs,wids=[],[],[],[]
        for wid in well_ids:
            p=data_dir/f'{wid}__horizontal_well.csv'
            try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
            except: continue
            if len(df)==0: continue
            ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
        self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
        self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids)
        self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
        self.tree=cKDTree(self.xy/self.scale)

    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
        dist,idx=self.tree.query(q,k=nf,workers=-1)
        if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
        ord_=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
        dk=np.take_along_axis(dist,ord_,1); ik=np.take_along_axis(idx,ord_,1)
        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
        sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
        ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
        var=((an-ap[:,None])**2*w).sum(1)/safe
        return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)


# =============================================================================
# SECTION 5: Feature Builder
# =============================================================================

# Feature column order for the 56-dim model input
FEATURE_NAMES = [
    # Group 1: Raw signals (6)
    "gr", "gr_d1", "gr_env", "dz_dmd", "dx_dmd", "dy_dmd",
    # Group 2: Physical priors - delta (30)
    "beam_cons_d", "beam_sm5_d", "beam_cons_w_d", "beam_mean_d", "beam_std_d",
    "pf_ancc_d", "pf_ens_d", "pf_z_d", "pf_std",
    "sc8_d", "sc15_d", "sc25_d", "sc_ens_d", "hyb_d",
    "dtw_ens_d", "dtw_stoch_mean_d", "dtw_r50_d", "dtw_r100_d", "dtw_stoch_std", "dtw_slope_mean",
    "form_mean_d", "form_std_d", "tvtF_ANCC_d", "tvtF_BUDA_d",
    "tvt_dense_d", "tvt_densew_d",
    "likpf_s3_d", "likpf_s5_d", "likpf_s8_d", "likpf_s12_d",
    # Group 3: TD misfit (9)
    "tdbc_n5", "tdbc_0", "tdbc_5",
    "tdpf_n5", "tdpf_0", "tdpf_5",
    "tddtw_n5", "tddtw_0", "tddtw_5",
    # Group 4: Cross-algorithm disagreement (6)
    "dtw_vs_beam", "dtw_vs_pf", "sc_vs_beam", "pf_ens_vs_single", "sig_std", "dtw_stoch_cv",
    # Group 5: Position & geometry (5)
    "md_since", "frac", "z_rel", "dxy", "azimuth",
]
assert len(FEATURE_NAMES) == 56

_FI = None
_DI = None


def build_well_features(hw_path, tw_path, is_train):
    """Build 56-dim feature matrix + delta target for one well.
    Returns: dict with keys 'wid', 'features' [N, 56], 'target' [N] or None, 'last_tvt', 'md_since'.
    Returns None if well is invalid.
    """
    global _FI, _DI
    wid = Path(hw_path).stem.replace('__horizontal_well', '')
    _well_seed = int(hashlib.md5(wid.encode()).hexdigest(), 16) % (2**31)
    np.random.seed(SEED + _well_seed)
    try:
        hw = pd.read_csv(hw_path)
        tw = pd.read_csv(tw_path).sort_values('TVT')
    except Exception:
        return None
    if is_train and 'TVT' not in hw.columns:
        return None
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0 or len(kn) < 10:
        return None
    if is_train and hw['TVT'].isna().all():
        return None
    tw_tvt = tw['TVT'].to_numpy(np.float32)
    tw_gr = tw['GR'].fillna(tw['GR'].mean()).to_numpy(np.float32)
    if len(tw_tvt) < 3:
        return None

    lk = kn.iloc[-1]
    last_tvt = float(lk['TVT_input'])
    nh = len(ev)

    # --- Compute physical priors ---
    _jit_seed_a = (SEED + _well_seed) % (2**31)
    _jit_seed_z = (SEED + _well_seed + 1) % (2**31)
    _jit_seed_dtw = (SEED + _well_seed + 2) % (2**31)

    # PF-ANCC + PF-Z + PF-ensemble
    pf_a, std_a = run_pf_ancc(hw, tw_tvt, tw_gr, jit_seed=_jit_seed_a)
    if len(pf_a) == 0:
        return None
    pf_z_pred, _ = run_pf_z(hw, tw_tvt, tw_gr, jit_seed=_jit_seed_z)
    has_z = len(pf_z_pred) == len(pf_a) and not np.any(np.isnan(pf_z_pred))
    pf_ens, pf_ens_std, _ = run_pf_lik_ensemble(hw, tw_tvt, tw_gr, base_seed=_jit_seed_a)

    # GR processing
    gr_full = hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr = gr_full.iloc[ev.index[0]:].to_numpy(np.float32)[:nh]
    kgr = gr_full.iloc[:len(kn)].to_numpy(np.float32)

    # Beam search (3 key configs + weighted)
    bpaths = {}
    for (bs, mc, es, r, tag) in BEAMS:
        bpaths[tag] = beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
    bpaths['cons_w'] = beam_search_weighted(hgr, tw_tvt, tw_gr, last_tvt, 10, 20.0, 144.0, 2, alpha=0.3)
    beam_ref = (bpaths['cons'] + bpaths['sm5']) / 2.

    # NCC
    ktvt = kn['TVT_input'].to_numpy(np.float32)
    sc_res, sc_ens = multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
    sc8, _ = sc_res[0]; sc15, _ = sc_res[1]; sc25, _ = sc_res[2]
    sc_trust = float(np.clip(len(kn) / 200., 0., 0.6))
    hyb_ref = (1 - sc_trust) * beam_ref + sc_trust * sc_ens

    # DTW multiscale + stochastic
    full_gr = gr_full.values.astype(np.float32)
    dtw_tvts, dtw_slopes, _, dtw_ens_ev_full = run_dtw_multiscale(full_gr, tw_tvt, tw_gr, last_tvt)
    dtw_mean_stoch, dtw_std_stoch, dtw_cv_stoch = run_dtw_stochastic(
        full_gr, tw_tvt, tw_gr, last_tvt, radius=50, K=DTW_STOCH_K,
        temperature=DTW_STOCH_TEMP, jit_seed=_jit_seed_dtw)

    ev_start = ev.index[0]
    def _ev_slice(arr):
        return arr[ev_start:ev_start+nh].astype(np.float32)

    dtw_ens_ev = _ev_slice(dtw_ens_ev_full)
    dtw_mean_ev = _ev_slice(dtw_mean_stoch)
    dtw_std_ev = _ev_slice(dtw_std_stoch)
    dtw_cv_ev = _ev_slice(dtw_cv_stoch)
    dtw_r50_ev = _ev_slice(dtw_tvts[50])
    dtw_r100_ev = _ev_slice(dtw_tvts[100])
    dtw_slope_mean_ev = np.stack([_ev_slice(dtw_slopes[r]) for r in DTW_RADII], 1).mean(1).astype(np.float32)

    # Formation / Dense ANCC
    swid = wid if is_train else None
    xy_ev = ev[["X", "Y"]].to_numpy(np.float64)
    xy_kn = kn[["X", "Y"]].to_numpy(np.float64)
    form_ev, _ = _FI.impute(xy_ev, self_wid=swid)
    form_kn, _ = _FI.impute(xy_kn, self_wid=swid)
    z_kn = kn['Z'].to_numpy(np.float32)
    z_ev = ev['Z'].to_numpy(np.float32)

    # Formation ANCC and BUDA specifically
    fi_ancc = FORMATIONS.index("ANCC")
    fi_buda = FORMATIONS.index("BUDA")
    b_ancc, _, _, _, _ = seg_b_well(ktvt, z_kn, form_kn[:, fi_ancc])
    b_buda, _, _, _, _ = seg_b_well(ktvt, z_kn, form_kn[:, fi_buda])
    tvtF_ANCC = (-z_ev + form_ev[:, fi_ancc] + b_ancc).astype(np.float32)
    tvtF_BUDA = (-z_ev + form_ev[:, fi_buda] + b_buda).astype(np.float32)

    # Formation mean/std across all formations
    form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, _, _, _, _ = seg_b_well(ktvt, z_kn, form_kn[:, fi2])
        tvt_f = (-z_ev + form_ev[:, fi2] + b_full).astype(np.float32)
        form_list.append(tvt_f)
    fs = np.stack(form_list, 1)
    form_mean_d = (fs.mean(1) - last_tvt).astype(np.float32)
    form_std_d = fs.std(1).astype(np.float32)

    # Dense ANCC
    d_ancc, _, _ = _DI.impute(xy_ev, self_wid=swid)
    d_kn, _, _ = _DI.impute(xy_kn, self_wid=swid)
    b_d = float(np.median(ktvt + z_kn - d_kn))
    tvt_dense = (-z_ev + d_ancc + b_d).astype(np.float32)
    _, _, _, _, b_dw = seg_b_well(ktvt, z_kn, d_kn)
    tvt_densew = (-z_ev + d_ancc + b_dw).astype(np.float32)

    # lik-PF (128-seed likelihood-weighted particle filter)
    likpf_out, _ = lik_pf(hw, tw, seed_base=_well_seed)
    likpf_s3 = likpf_out.get("pf_scale_3", np.full(nh, last_tvt, np.float32))
    likpf_s5 = likpf_out.get("pf_scale_5", np.full(nh, last_tvt, np.float32))
    likpf_s8 = likpf_out.get("pf_scale_8", np.full(nh, last_tvt, np.float32))
    likpf_s12 = likpf_out.get("pf_scale_12", np.full(nh, last_tvt, np.float32))

    # --- Derived quantities ---
    hmd = ev['MD'].to_numpy(np.float32)
    md_since = hmd - float(lk['MD'])
    frac = (np.arange(nh) / max(nh - 1, 1)).astype(np.float32)

    # GR derivatives
    gr_s = pd.Series(gr_full.values)
    gr_d1 = gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env = gr_s.rolling(21, center=True, min_periods=1).max().iloc[ev.index].values.astype(np.float32)

    # Trajectory
    mdd = hw['MD'].diff().replace(0, np.nan)
    dzdmd = (hw['Z'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd = (hw['X'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dydmd = (hw['Y'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dxy = np.sqrt((ev['X'].values - float(lk['X']))**2 + (ev['Y'].values - float(lk['Y']))**2).astype(np.float32)
    _dx_diff = hw['X'].diff().iloc[ev.index].values.astype(np.float64)
    _dy_diff = hw['Y'].diff().iloc[ev.index].values.astype(np.float64)
    azimuth = np.arctan2(_dy_diff, _dx_diff).astype(np.float32)
    z_rel = (z_ev - float(lk['Z'])).astype(np.float32)

    # Ensemble signals for sig_std
    all_sigs = [pf_a, bpaths['cons'], bpaths['sm5'], sc_ens, tvtF_ANCC, tvt_dense, dtw_ens_ev]
    sig_mat = np.stack(all_sigs, 1)
    sig_std = sig_mat.std(1).astype(np.float32)

    # --- Assemble 56-dim feature matrix ---
    # All physical priors in delta space (subtract last_tvt)
    pf_ens_use = pf_ens if len(pf_ens) > 0 else np.full(nh, last_tvt, np.float32)
    pf_ens_std_use = pf_ens_std if len(pf_ens_std) > 0 else np.zeros(nh, np.float32)
    pf_z_use = pf_z_pred.astype(np.float32) if has_z else np.full(nh, last_tvt, np.float32)

    # TD misfit features: GR - tw_gr(pred + offset)
    def _td(pred_arr, offset):
        return hgr - np.interp(pred_arr + offset, tw_tvt, tw_gr).astype(np.float32)

    feat_matrix = np.column_stack([
        # Group 1: Raw (6)
        hgr, gr_d1, gr_env, dzdmd, dxdmd, dydmd,
        # Group 2: Physical priors delta (30)
        bpaths['cons'] - last_tvt, bpaths['sm5'] - last_tvt, bpaths['cons_w'] - last_tvt,
        np.stack([p - last_tvt for p in bpaths.values()], 1).mean(1),
        np.stack([p - last_tvt for p in bpaths.values()], 1).std(1),
        pf_a - last_tvt, pf_ens_use - last_tvt, pf_z_use - last_tvt, std_a,
        sc8 - last_tvt, sc15 - last_tvt, sc25 - last_tvt, sc_ens - last_tvt, hyb_ref - last_tvt,
        dtw_ens_ev - last_tvt, dtw_mean_ev - last_tvt, dtw_r50_ev - last_tvt, dtw_r100_ev - last_tvt,
        dtw_std_ev, dtw_slope_mean_ev,
        form_mean_d, form_std_d, tvtF_ANCC - last_tvt, tvtF_BUDA - last_tvt,
        tvt_dense - last_tvt, tvt_densew - last_tvt,
        likpf_s3 - last_tvt, likpf_s5 - last_tvt, likpf_s8 - last_tvt, likpf_s12 - last_tvt,
        # Group 3: TD misfit (9)
        _td(beam_ref, -5.), _td(beam_ref, 0.), _td(beam_ref, 5.),
        _td(pf_a, -5.), _td(pf_a, 0.), _td(pf_a, 5.),
        _td(dtw_ens_ev, -5.), _td(dtw_ens_ev, 0.), _td(dtw_ens_ev, 5.),
        # Group 4: Cross-disagreement (6)
        dtw_ens_ev - bpaths['cons'], dtw_ens_ev - pf_a,
        sc_ens - bpaths['cons'], (pf_ens_use - pf_a),
        sig_std, dtw_cv_ev,
        # Group 5: Position & geometry (5)
        md_since, frac, z_rel, dxy, azimuth,
    ]).astype(np.float32)

    # Replace NaN/Inf with 0
    feat_matrix = np.nan_to_num(feat_matrix, nan=0.0, posinf=0.0, neginf=0.0)

    assert feat_matrix.shape == (nh, 56), f"Expected ({nh}, 56), got {feat_matrix.shape}"

    result = {
        'wid': wid,
        'features': feat_matrix,
        'last_tvt': last_tvt,
        'md_since': md_since,
        'n_eval': nh,
    }
    if is_train:
        true_tvt = ev['TVT'].to_numpy(np.float32)
        target = true_tvt - np.float32(last_tvt)
        result['target'] = target

    return result


# =============================================================================
# SECTION 6: WellSeqDataset (Sliding Window + Padding + Mask)
# =============================================================================

class WellSeqDataset(Dataset):
    """
    Dataset that creates sliding windows from per-well feature matrices.

    Each item is a tuple (features, target, mask, positions):
      - features: [W, 56] float32
      - target:   [W] float32
      - mask:     [W] float32 (1=valid, 0=padding)
      - positions: [W] float32 (md_since values for PE)
    """

    def __init__(self, well_data_list, window_size=512, stride=256, mean=None, std=None):
        """
        well_data_list: list of dicts from build_well_features (must have 'features', 'target', 'md_since')
        mean, std: per-feature normalization params [56]. If None, computed from data.
        """
        self.W = window_size
        self.stride = stride

        # Collect all features for normalization
        all_feats = [d['features'] for d in well_data_list]

        if mean is None or std is None:
            cat = np.concatenate(all_feats, axis=0)
            self.mean = cat.mean(axis=0).astype(np.float32)
            self.std = np.maximum(cat.std(axis=0), 1e-6).astype(np.float32)
        else:
            self.mean = mean.astype(np.float32)
            self.std = std.astype(np.float32)

        # Build windows
        self.windows = []  # (features_W, target_W, mask_W, positions_W)
        for d in well_data_list:
            feat = (d['features'] - self.mean) / self.std  # Normalize
            target = d['target']
            md_s = d['md_since']
            L = len(feat)

            if L <= self.W:
                # Pad to W
                pad_len = self.W - L
                feat_pad = np.pad(feat, ((0, pad_len), (0, 0)), mode='constant', constant_values=0.)
                tgt_pad = np.pad(target, (0, pad_len), mode='constant', constant_values=0.)
                mask = np.zeros(self.W, dtype=np.float32)
                mask[:L] = 1.0
                pos_pad = np.pad(md_s, (0, pad_len), mode='constant', constant_values=0.)
                self.windows.append((feat_pad, tgt_pad, mask, pos_pad))
            else:
                # Sliding windows
                starts = list(range(0, L - self.W + 1, self.stride))
                if starts[-1] + self.W < L:
                    starts.append(L - self.W)
                for s in starts:
                    self.windows.append((
                        feat[s:s+self.W],
                        target[s:s+self.W],
                        np.ones(self.W, dtype=np.float32),
                        md_s[s:s+self.W],
                    ))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        feat, tgt, mask, pos = self.windows[idx]
        return (
            torch.from_numpy(feat),
            torch.from_numpy(tgt),
            torch.from_numpy(mask),
            torch.from_numpy(pos),
        )


# =============================================================================
# SECTION 7: Model Verification (forward pass test)
# =============================================================================

def verify_model_forward():
    """Quick test: create UNet model, random input, verify output shape."""
    print("=" * 60)
    print("Verification: SCA-U2Net-Reg Forward Pass")
    print("=" * 60)
    model = SCA_U2Net_Reg(UCFG).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model parameters: {n_params:,}")
    print(f"  Device: {DEVICE}")

    B, L = 4, 512
    x = torch.randn(B, UCFG.in_ch, L, device=DEVICE)  # channel-first: [B, 56, L]
    mask = torch.ones(B, L, device=DEVICE)
    mask[0, 400:] = 0

    with torch.no_grad():
        fuse, sides = model(x)

    assert fuse.shape == (B, L), f"Expected ({B}, {L}), got {fuse.shape}"
    print(f"  Input shape:  [{B}, {UCFG.in_ch}, {L}]")
    print(f"  Fuse output shape: [{B}, {L}]")
    print(f"  Side outputs: {len(sides)}")
    print(f"  Output range: [{fuse.min().item():.4f}, {fuse.max().item():.4f}]")

    # Test backward with deep supervision
    target = torch.randn(B, L, device=DEVICE)
    loss = deep_supervision_rmse(fuse, sides, target, mask)
    loss.backward()
    print(f"  Loss (random): {loss.item():.4f}")
    print("  Backward pass: OK")
    print("  [PASS] UNet model forward/backward verified.\n")
    return True


def verify_feature_builder(train_dir):
    """Test feature builder on first available well."""
    print("=" * 60)
    print("Verification: Feature Builder")
    print("=" * 60)
    global _FI, _DI
    hw_paths = sorted(train_dir.glob('*__horizontal_well.csv'))
    if not hw_paths:
        print("  [SKIP] No training data found.")
        return False

    # Build spatial imputers
    all_wids = [p.stem.replace('__horizontal_well', '') for p in hw_paths]
    print(f"  Building spatial imputers ({len(all_wids)} wells)...")
    _FI = FormationPlaneKNN(all_wids, train_dir)
    _DI = DenseANCCImputer(all_wids, train_dir)

    # Test on first well
    hw_p = hw_paths[0]
    tw_p = train_dir / f'{hw_p.stem.replace("__horizontal_well", "")}__typewell.csv'
    if not tw_p.exists():
        print("  [SKIP] No matching typewell.")
        return False

    t0 = time.time()
    result = build_well_features(str(hw_p), str(tw_p), is_train=True)
    elapsed = time.time() - t0

    if result is None:
        print("  [SKIP] Well returned None (insufficient data).")
        return False

    feat = result['features']
    target = result['target']
    print(f"  Well: {result['wid']}")
    print(f"  Feature matrix shape: {feat.shape} (expected (N, 56))")
    print(f"  Target shape: {target.shape}")
    print(f"  NaN count in features: {np.isnan(feat).sum()}")
    print(f"  Feature range: [{feat.min():.2f}, {feat.max():.2f}]")
    print(f"  Target range: [{target.min():.2f}, {target.max():.2f}]")
    print(f"  Time: {elapsed:.1f}s")
    assert feat.shape[1] == 56, f"Expected 56 features, got {feat.shape[1]}"
    assert len(target) == feat.shape[0]
    print("  [PASS] Feature builder verified.\n")
    return True


# =============================================================================
# SECTION 8: Training Loop + OOF Inference
# =============================================================================

def _build_one_well_safe(hw_path, tw_path):
    """Thread-safe wrapper for build_well_features (for parallel execution)."""
    try:
        return build_well_features(hw_path, tw_path, is_train=True)
    except Exception:
        return None


def build_all_features(train_dir, cache_path=None, n_jobs=None):
    """Build features for all training wells. Uses cache if available.
    Supports parallel execution via joblib for speed.
    """
    global _FI, _DI
    if cache_path and Path(cache_path).exists():
        print(f"  Loading cached features from {cache_path}...")
        return joblib.load(cache_path)

    hw_paths = sorted(train_dir.glob('*__horizontal_well.csv'))
    all_wids = [p.stem.replace('__horizontal_well', '') for p in hw_paths]
    print(f"  Building spatial imputers ({len(all_wids)} wells)...")
    _FI = FormationPlaneKNN(all_wids, train_dir)
    _DI = DenseANCCImputer(all_wids, train_dir)

    # Prepare argument pairs
    args = []
    for hw_p in hw_paths:
        wid = hw_p.stem.replace('__horizontal_well', '')
        tw_p = train_dir / f'{wid}__typewell.csv'
        if tw_p.exists():
            args.append((str(hw_p), str(tw_p)))

    # n_jobs: high on server (56 cores), serial on Kaggle (4 cores, test only 3 wells anyway)
    if n_jobs is None:
        n_jobs = NCPU
    print(f"  Building features for {len(args)} wells (n_jobs={n_jobs})...")
    t0 = time.time()

    if n_jobs <= 1:
        # Serial fallback (deterministic, used on Kaggle)
        results = []
        for i, (hp, tp) in enumerate(args):
            r = build_well_features(hp, tp, is_train=True)
            if r is not None:
                results.append(r)
            if (i + 1) % 50 == 0:
                elapsed = time.time() - t0
                eta = elapsed / (i + 1) * (len(args) - i - 1)
                print(f"    [{i+1}/{len(args)}] built {len(results)} wells, "
                      f"elapsed={elapsed:.0f}s, ETA={eta:.0f}s")
    else:
        # multiprocessing + fork (Linux): child processes inherit _FI/_DI globals
        # without pickling. PF core is numba (nogil), but pandas/beam/DTW and other
        # Python stages still benefit from process-level parallelism.
        raw = Parallel(n_jobs=n_jobs, backend="multiprocessing", verbose=10)(
            delayed(_build_one_well_safe)(hp, tp) for hp, tp in args
        )
        results = [r for r in raw if r is not None]

    print(f"  Total: {len(results)} wells built in {time.time()-t0:.0f}s")
    if cache_path:
        joblib.dump(results, cache_path, compress=3)
        print(f"  Cached to {cache_path}")
    return results


def predict_well_oof(model, well_data, norm_mean, norm_std, device, window_size=512, stride=256):
    """Sliding-window inference for SCA-U2Net-Reg (channel-first input).
    Returns: predicted delta array [N_eval].
    """
    model.eval()
    feat_raw = well_data['features']
    L = len(feat_raw)

    # Normalize and transpose to [56, L] for channel-first model
    feat = ((feat_raw - norm_mean) / norm_std).astype(np.float32)
    feat_t = feat.T  # [56, L]

    if L <= window_size:
        pad_len = window_size - L
        feat_pad = np.pad(feat_t, ((0, 0), (0, pad_len)), mode='constant')  # [56, W]

        with torch.no_grad():
            x = torch.from_numpy(feat_pad).unsqueeze(0).to(device)  # [1, 56, W]
            fuse, _ = model(x)  # fuse: [1, W]
        return fuse[0, :L].cpu().numpy()

    starts = list(range(0, L - window_size + 1, stride))
    if starts[-1] + window_size < L:
        starts.append(L - window_size)

    pred_sum = np.zeros(L, dtype=np.float64)
    pred_cnt = np.zeros(L, dtype=np.float64)

    VAL_BATCH = 16
    with torch.no_grad():
        for bi in range(0, len(starts), VAL_BATCH):
            batch_starts = starts[bi:bi+VAL_BATCH]
            bs = len(batch_starts)
            # Stack windows: [bs, 56, W]
            x_batch = np.stack([feat_t[:, s:s+window_size] for s in batch_starts])
            x = torch.from_numpy(x_batch).to(device)
            fuse, _ = model(x)  # fuse: [bs, W]
            out_np = fuse.cpu().numpy()
            for j, s in enumerate(batch_starts):
                pred_sum[s:s+window_size] += out_np[j]
                pred_cnt[s:s+window_size] += 1.0

    return (pred_sum / np.maximum(pred_cnt, 1.0)).astype(np.float32)


def train_one_fold(fold, train_data, val_data, norm_mean, norm_std, device, cfg_t=None, cfg_u=None):
    """Train SCA-U2Net-Reg for one fold. Returns (model, val_rmse, val_predictions_dict)."""
    cfg_t = cfg_t or TRCFG
    cfg_u = cfg_u or UCFG

    # Fix per-fold seed for reproducible weight init + shuffle order
    fold_seed = cfg_t.seed + fold * 1000
    torch.manual_seed(fold_seed)
    np.random.seed(fold_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)

    # Build datasets
    train_ds = WellSeqDataset(train_data, cfg_t.window_size, cfg_t.stride, norm_mean, norm_std)
    g = torch.Generator()
    g.manual_seed(fold_seed)
    train_dl = DataLoader(train_ds, batch_size=cfg_t.batch_size, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True, generator=g)

    # Build UNet model
    model = SCA_U2Net_Reg(cfg_u).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg_t.lr, weight_decay=cfg_t.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg_t.epochs, eta_min=1e-6)

    best_val_rmse = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(cfg_t.epochs):
        # --- Training ---
        model.train()
        train_loss_sum = 0.0
        train_n = 0
        for batch in train_dl:
            feat_b, tgt_b, mask_b, pos_b = [t.to(device) for t in batch]
            optimizer.zero_grad()
            # Transpose features: [B, W, 56] -> [B, 56, W] for channel-first UNet
            x_ch = feat_b.transpose(1, 2)
            fuse, sides = model(x_ch)
            loss = deep_supervision_rmse(fuse, sides, tgt_b, mask_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss_sum += loss.item() * mask_b.sum().item()
            train_n += mask_b.sum().item()

        scheduler.step()
        train_rmse = train_loss_sum / max(train_n, 1)

        # --- Validation (full-well sliding window) ---
        model.eval()
        val_preds = {}
        all_true = []
        all_pred = []
        for d in val_data:
            pred_delta = predict_well_oof(model, d, norm_mean, norm_std, device,
                                          cfg_t.window_size, cfg_t.stride)
            val_preds[d['wid']] = pred_delta
            all_true.append(d['target'])
            all_pred.append(pred_delta)

        all_true_cat = np.concatenate(all_true)
        all_pred_cat = np.concatenate(all_pred)
        val_rmse = float(np.sqrt(np.mean((all_true_cat - all_pred_cat) ** 2)))

        # Early stopping
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0 or patience_counter == 0:
            lr_now = scheduler.get_last_lr()[0]
            print(f"    [UNet] Fold {fold} Epoch {epoch+1:02d}: "
                  f"train_rmse={train_rmse:.4f}, val_rmse={val_rmse:.4f}, "
                  f"best={best_val_rmse:.4f}, lr={lr_now:.2e}, pat={patience_counter}")

        if patience_counter >= cfg_t.patience:
            print(f"    [UNet] Early stop at epoch {epoch+1}")
            break

    # Restore best model
    model.load_state_dict(best_state)
    model.eval()

    # Re-compute val predictions with best model
    val_preds_best = {}
    for d in val_data:
        pred_delta = predict_well_oof(model, d, norm_mean, norm_std, device,
                                      cfg_t.window_size, cfg_t.stride)
        val_preds_best[d['wid']] = pred_delta

    return model, best_val_rmse, val_preds_best


def run_cv_training(all_well_data, device, cfg_t=None, cfg_u=None):
    """Run full 5-fold CV training for SCA-U2Net-Reg with fold-level resume support.
    If a fold checkpoint already exists, loads it instead of retraining.
    Returns (models, oof_predictions, global_mean, global_std, cv_rmse_tvt).
    """
    cfg_t = cfg_t or TRCFG
    cfg_u = cfg_u or UCFG
    n_splits = cfg_t.n_splits

    # Group wells by wid for GroupKFold
    wids = np.array([d['wid'] for d in all_well_data])

    gkf = GroupKFold(n_splits=n_splits)
    indices = np.arange(len(all_well_data))

    # Compute global normalization stats from ALL data
    all_feats = np.concatenate([d['features'] for d in all_well_data], axis=0)
    global_mean = all_feats.mean(axis=0).astype(np.float32)
    global_std = np.maximum(all_feats.std(axis=0), 1e-6).astype(np.float32)
    del all_feats
    gc.collect()
    print(f"  Global normalization: mean_range=[{global_mean.min():.2f}, {global_mean.max():.2f}], "
          f"std_range=[{global_std.min():.4f}, {global_std.max():.2f}]")

    # Save normalization stats early (needed for resume)
    model_dir = Path(MODEL_DIR)
    model_dir.mkdir(exist_ok=True)
    joblib.dump({'mean': global_mean, 'std': global_std}, NORM_FILE)

    models = []
    oof_preds = {}  # wid -> predicted delta
    fold_rmses = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(indices, groups=wids)):
        print(f"\n  --- [UNet] Fold {fold} ({len(train_idx)} train, {len(val_idx)} val wells) ---")
        train_data = [all_well_data[i] for i in train_idx]
        val_data = [all_well_data[i] for i in val_idx]

        ckpt_path = model_dir / f"sca_u2net_fold{fold}.pt"

        # Fold-level resume: skip training if checkpoint exists
        if ckpt_path.exists():
            print(f"    [RESUME] Loading existing UNet checkpoint: {ckpt_path}")
            model = SCA_U2Net_Reg(cfg_u).to(device)
            model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
            model.eval()

            # Re-compute val predictions from loaded model
            val_preds = {}
            all_true, all_pred = [], []
            for d in val_data:
                pred_delta = predict_well_oof(model, d, global_mean, global_std, device,
                                              cfg_t.window_size, cfg_t.stride)
                val_preds[d['wid']] = pred_delta
                all_true.append(d['target'])
                all_pred.append(pred_delta)
            val_rmse = float(np.sqrt(np.mean(
                (np.concatenate(all_true) - np.concatenate(all_pred)) ** 2)))
            print(f"    [RESUME] Fold {fold} val_rmse = {val_rmse:.4f}")
        else:
            model, val_rmse, val_preds = train_one_fold(
                fold, train_data, val_data, global_mean, global_std, device, cfg_t, cfg_u
            )
            torch.save(model.state_dict(), ckpt_path)
            print(f"    [UNet] Fold {fold} val_rmse = {val_rmse:.4f} (saved)")

        models.append(model)
        fold_rmses.append(val_rmse)
        oof_preds.update(val_preds)

    # Compute overall CV RMSE
    all_true = []
    all_pred = []
    for d in all_well_data:
        wid = d['wid']
        if wid in oof_preds:
            all_true.append(d['target'])
            all_pred.append(oof_preds[wid])
    all_true_cat = np.concatenate(all_true)
    all_pred_cat = np.concatenate(all_pred)
    cv_rmse_delta = float(np.sqrt(np.mean((all_true_cat - all_pred_cat) ** 2)))

    # Convert delta to TVT for final RMSE
    all_true_tvt = []
    all_pred_tvt = []
    for d in all_well_data:
        wid = d['wid']
        if wid in oof_preds:
            last_tvt = d['last_tvt']
            all_true_tvt.append(d['target'] + last_tvt)
            all_pred_tvt.append(oof_preds[wid] + last_tvt)
    all_true_tvt_cat = np.concatenate(all_true_tvt)
    all_pred_tvt_cat = np.concatenate(all_pred_tvt)
    cv_rmse_tvt = float(np.sqrt(np.mean((all_true_tvt_cat - all_pred_tvt_cat) ** 2)))

    print(f"\n  {'='*60}")
    print(f"  SCA-U2Net CV Results:")
    print(f"    Fold RMSEs (delta): {[f'{r:.4f}' for r in fold_rmses]}")
    print(f"    CV RMSE (delta):    {cv_rmse_delta:.4f}")
    print(f"    CV RMSE (TVT):      {cv_rmse_tvt:.4f}")
    print(f"  {'='*60}")

    return models, oof_preds, global_mean, global_std, cv_rmse_tvt


# =============================================================================
# SECTION 9: Post-processing + Test Inference + Submission
# =============================================================================

def postprocess_predictions(pred_delta, md_since, sg_win=61, sg_poly=3, apply_sg=True):
    """Optional Savitzky-Golay smoothing on predicted delta."""
    if not apply_sg or len(pred_delta) < sg_win:
        return pred_delta
    try:
        smoothed = savgol_filter(pred_delta, sg_win, sg_poly).astype(np.float32)
        return smoothed
    except Exception:
        return pred_delta


def predict_test_well(models, well_data, norm_mean, norm_std, device,
                      window_size=512, stride=256, apply_sg=True):
    """Run 5-fold ensemble inference on one test well.
    Averages predictions from all fold models, then optionally applies SG smoothing.
    Returns: predicted TVT array [N_eval].
    """
    n_models = len(models)
    feat_raw = well_data['features']
    md_since = well_data['md_since']
    last_tvt = well_data['last_tvt']
    L = len(feat_raw)

    # Accumulate predictions from all folds
    all_fold_preds = np.zeros((n_models, L), dtype=np.float64)
    for fi, model in enumerate(models):
        model.to(device)
        pred_delta = predict_well_oof(model, well_data, norm_mean, norm_std, device,
                                      window_size, stride)
        all_fold_preds[fi] = pred_delta
        model.cpu()  # free GPU memory

    # Average across folds
    ensemble_delta = all_fold_preds.mean(axis=0).astype(np.float32)

    # Post-process
    ensemble_delta = postprocess_predictions(ensemble_delta, md_since, apply_sg=apply_sg)

    # Convert to TVT
    pred_tvt = ensemble_delta + last_tvt
    return pred_tvt, ensemble_delta


def build_test_features(test_dir, train_dir):
    """Build features for test wells. Handles visible wells (same wid in train/test)."""
    global _FI, _DI
    test_paths = sorted(test_dir.glob('*__horizontal_well.csv'))
    train_wids = set(p.stem.replace('__horizontal_well', '') for p in
                     train_dir.glob('*__horizontal_well.csv'))

    # Build spatial imputers from training data if not already built
    if _FI is None or _DI is None:
        hw_paths = sorted(train_dir.glob('*__horizontal_well.csv'))
        all_wids = [p.stem.replace('__horizontal_well', '') for p in hw_paths]
        print(f"  Building spatial imputers ({len(all_wids)} wells)...")
        _FI = FormationPlaneKNN(all_wids, train_dir)
        _DI = DenseANCCImputer(all_wids, train_dir)

    results = []
    for p in test_paths:
        wid = p.stem.replace('__horizontal_well', '')
        tw_p = test_dir / f'{wid}__typewell.csv'

        # Visible well: use train typewell if test typewell doesn't exist
        if not tw_p.exists() and wid in train_wids:
            tw_p = train_dir / f'{wid}__typewell.csv'
        if not tw_p.exists():
            print(f"  [WARN] No typewell for {wid}, skipping")
            continue

        # For visible wells, use train's TVT_input for better known section
        hw_test = pd.read_csv(p)
        if wid in train_wids:
            hw_train = pd.read_csv(train_dir / f'{wid}__horizontal_well.csv')
            hw_test['TVT_input'] = hw_train['TVT_input'].values
            tw_p = train_dir / f'{wid}__typewell.csv'

        # Write temporary hw with correct naming for build_well_features wid extraction
        tmp_path = str(Path('.') / f'{wid}__horizontal_well.csv')
        hw_test.to_csv(tmp_path, index=False)

        r = build_well_features(tmp_path, str(tw_p), is_train=False)
        os.unlink(tmp_path)

        if r is not None:
            r['wid'] = wid  # ensure correct wid
            ev = hw_test[hw_test['TVT_input'].isna()]
            r['ev_indices'] = ev.index.tolist()
            results.append(r)
            print(f"  Test well {wid}: {r['n_eval']} eval points")
        else:
            print(f"  [WARN] Feature build failed for {wid}")

    return results


def generate_submission(test_well_results, test_predictions, test_dir, output_path="submission.csv"):
    """Generate submission CSV from test predictions."""
    rows = []
    for wd, pred_tvt in zip(test_well_results, test_predictions):
        wid = wd['wid']
        ev_indices = wd['ev_indices']
        for i, row_idx in enumerate(ev_indices):
            rid = f"{wid}_{row_idx}"
            rows.append({'id': rid, 'tvt': float(pred_tvt[i])})

    sub_df = pd.DataFrame(rows)

    # Align with sample submission
    sample_path = CFG.dataset_path / "sample_submission.csv"
    if sample_path.exists():
        sample = pd.read_csv(sample_path)
        sub_df = sample[['id']].merge(sub_df, on='id', how='left')
        n_missing = sub_df['tvt'].isna().sum()
        if n_missing > 0:
            print(f"  [WARN] {n_missing} missing predictions, filling with median")
            sub_df['tvt'] = sub_df['tvt'].fillna(sub_df['tvt'].median())
    else:
        print("  [INFO] sample_submission.csv not found, using raw predictions")

    sub_df[['id', 'tvt']].to_csv(output_path, index=False)
    print(f"  Submission saved: {output_path} ({len(sub_df)} rows)")
    return sub_df


def run_test_inference(models, norm_mean, norm_std, device, train_dir, test_dir,
                       window_size=512, stride=256, apply_sg=True):
    """Full test inference pipeline."""
    print("\n─── Test Inference ───")

    # Build test features
    print("  Building test well features...")
    test_well_data = build_test_features(test_dir, train_dir)
    if not test_well_data:
        print("  [ERROR] No test wells processed")
        return None

    # Predict each test well
    print(f"  Predicting {len(test_well_data)} test wells (5-fold ensemble)...")
    test_predictions = []
    for wd in test_well_data:
        pred_tvt, pred_delta = predict_test_well(
            models, wd, norm_mean, norm_std, device, window_size, stride, apply_sg)
        test_predictions.append(pred_tvt)
        print(f"    {wd['wid']}: N={len(pred_tvt)}, "
              f"TVT=[{pred_tvt.min():.2f}, {pred_tvt.max():.2f}], "
              f"delta=[{pred_delta.min():.2f}, {pred_delta.max():.2f}]")

    # Generate submission
    sub_df = generate_submission(test_well_data, test_predictions, test_dir)
    return sub_df


# =============================================================================
# SECTION 10: Kaggle Inference Path (standalone model loading + inference)
# =============================================================================

def kaggle_inference():
    """Kaggle notebook inference: load saved UNet models + norm stats, predict test wells."""
    print("=" * 70)
    print(" SCA-U2Net-Reg Kaggle Inference Mode")
    print("=" * 70)

    train_dir = CFG.dataset_path / "train"
    test_dir = CFG.dataset_path / "test"

    # Load normalization stats
    norm_data = joblib.load(NORM_FILE)
    norm_mean, norm_std = norm_data['mean'], norm_data['std']
    print(f"  Loaded normalization stats from {NORM_FILE}")

    # Load 5 fold UNet models
    model_dir = Path(MODEL_DIR)
    models = []
    for fold in range(TRCFG.n_splits):
        model = SCA_U2Net_Reg(UCFG)
        ckpt_path = model_dir / f"sca_u2net_fold{fold}.pt"
        model.load_state_dict(torch.load(ckpt_path, map_location='cpu', weights_only=True))
        model.eval()
        models.append(model)
    print(f"  Loaded {len(models)} UNet fold models from {model_dir}/")

    # Run inference
    sub_df = run_test_inference(
        models, norm_mean, norm_std, DEVICE, train_dir, test_dir,
        TRCFG.window_size, TRCFG.stride, apply_sg=True
    )
    return sub_df


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # Kaggle inference mode: if models exist and IS_KAGGLE, just predict
    if IS_KAGGLE and Path(MODEL_DIR).exists() and Path(NORM_FILE).exists():
        kaggle_inference()
        raise SystemExit(0)

    print("\n" + "=" * 70)
    print(" SCA-U2Net-Reg")
    print(" Training + CV + Inference")
    print("=" * 70 + "\n")

    train_dir = CFG.dataset_path / "train"
    if not train_dir.exists():
        print(f"[ERROR] Training data not found at {train_dir}")
        raise SystemExit(1)

    # Step 1: Build (or load cached) features
    print("─── Step 1: Build Features ───")
    cache_path = FEAT_CACHE
    all_well_data = build_all_features(train_dir, cache_path=cache_path)
    print(f"  Total wells with valid features: {len(all_well_data)}")
    total_points = sum(d['features'].shape[0] for d in all_well_data)
    print(f"  Total eval points: {total_points:,}")

    # Step 2: 5-Fold CV Training
    print("\n─── Step 2: 5-Fold Cross-Validation Training (SCA-U2Net-Reg) ───")
    print(f"  Device: {DEVICE}")
    print(f"  Model: in_ch={UCFG.in_ch}, mid_ch={UCFG.mid_ch}, out_ch={UCFG.out_ch}")
    print(f"  Training: W={TRCFG.window_size}, stride={TRCFG.stride}, "
          f"bs={TRCFG.batch_size}, epochs={TRCFG.epochs}, "
          f"lr={TRCFG.lr}, patience={TRCFG.patience}")

    models, oof_preds, norm_mean, norm_std, cv_rmse = run_cv_training(
        all_well_data, DEVICE, TRCFG, UCFG
    )

    # Step 3: Save OOF predictions
    print("\n─── Step 3: Save OOF Predictions ───")
    oof_records = []
    for d in all_well_data:
        wid = d['wid']
        if wid in oof_preds:
            pred_delta = oof_preds[wid]
            pred_tvt = pred_delta + d['last_tvt']
            true_tvt = d['target'] + d['last_tvt']
            for i in range(len(pred_delta)):
                oof_records.append({
                    'wid': wid, 'idx': i,
                    'md_since': d['md_since'][i],
                    'true_delta': d['target'][i],
                    'pred_delta': pred_delta[i],
                    'true_tvt': true_tvt[i],
                    'pred_tvt': pred_tvt[i],
                })
    oof_df = pd.DataFrame(oof_records)
    oof_df.to_csv("sca_u2net_oof_predictions.csv", index=False)
    print(f"  Saved {len(oof_df)} OOF predictions to sca_u2net_oof_predictions.csv")

    # Per-well RMSE analysis
    print("\n─── Step 4: Per-Well Analysis ───")
    well_rmses = []
    for d in all_well_data:
        wid = d['wid']
        if wid in oof_preds:
            true_tvt = d['target'] + d['last_tvt']
            pred_tvt = oof_preds[wid] + d['last_tvt']
            rmse_w = float(np.sqrt(np.mean((true_tvt - pred_tvt) ** 2)))
            well_rmses.append((wid, rmse_w, len(d['target'])))

    well_rmses.sort(key=lambda x: x[1], reverse=True)
    print(f"  Worst 10 wells:")
    for wid, rmse_w, n in well_rmses[:10]:
        print(f"    {wid}: RMSE={rmse_w:.2f} (n={n})")
    print(f"  Best 5 wells:")
    for wid, rmse_w, n in well_rmses[-5:]:
        print(f"    {wid}: RMSE={rmse_w:.2f} (n={n})")

    # Hard wells monitoring
    HARD_WELLS = ["86454a6f", "fb03ae90", "1b1eba53", "389ae58f", "896d15b9"]
    hard_true, hard_pred = [], []
    for d in all_well_data:
        if d['wid'] in HARD_WELLS and d['wid'] in oof_preds:
            hard_true.append(d['target'] + d['last_tvt'])
            hard_pred.append(oof_preds[d['wid']] + d['last_tvt'])
    if hard_true:
        ht = np.concatenate(hard_true)
        hp = np.concatenate(hard_pred)
        hard_rmse = float(np.sqrt(np.mean((ht - hp) ** 2)))
        print(f"\n  Hard wells RMSE: {hard_rmse:.4f}")

    print(f"\n{'='*70}")
    print(f" CV RMSE (TVT): {cv_rmse:.4f}")
    print(f"{'='*70}")

    # Step 5: Test Inference + Submission
    test_dir = CFG.dataset_path / "test"
    if test_dir.exists():
        print("\n─── Step 5: Test Inference + Submission ───")
        sub_df = run_test_inference(
            models, norm_mean, norm_std, DEVICE, train_dir, test_dir,
            TRCFG.window_size, TRCFG.stride, apply_sg=True
        )
    else:
        print(f"\n[INFO] Test data not found at {test_dir}. Skipping inference.")

    # Final summary
    print(f"\n{'='*70}")
    print(f" SCA-U2Net-Reg Complete!")
    print(f"   CV RMSE (TVT): {cv_rmse:.4f}")
    print(f"   Models:        {MODEL_DIR}/sca_u2net_fold*.pt")
    print(f"   Norm stats:    {NORM_FILE}")
    print(f"   OOF CSV:       sca_u2net_oof_predictions.csv")
    if test_dir.exists():
        print(f"   Submission:    submission.csv")
    print(f"{'='*70}")


 SCA-U2Net-Reg Kaggle Inference Mode
  Loaded normalization stats from /kaggle/input/datasets/bioconvolutionyt/sca-u2net-artifacts/sca_u2net_norm_stats.pkl
  Loaded 5 UNet fold models from /kaggle/input/datasets/bioconvolutionyt/sca-u2net-artifacts/models_sca_u2net/

─── Test Inference ───
  Building test well features...
  Building spatial imputers (773 wells)...
  Test well 000d7d20: 3836 eval points
  Test well 00bbac68: 6014 eval points
  Test well 00e12e8b: 4301 eval points
  Predicting 3 test wells (5-fold ensemble)...
    000d7d20: N=3836, TVT=[11735.84, 11749.40], delta=[-11.53, 2.03]
    00bbac68: N=6014, TVT=[12206.24, 12242.77], delta=[-17.30, 19.23]
    00e12e8b: N=4301, TVT=[11597.79, 11610.24], delta=[-7.03, 5.42]
  Submission saved: submission.csv (14151 rows)


SystemExit: 0